# 🌿 EfficientNetB4 + MSA Cassava Disease Classifier
## Detection of Cassava Leaf Disease Classes
### Okobi Ebubechukwu Moses | 22CD009441 | Landmark University
### Supervisor: Dr. Odeyemi Temitope

---

## 📌 Before You Start — Read This First

This notebook implements the five-class classification pipeline that matches the current dataset and is updated to use GPU compute when available.
Run every cell one at a time, top to bottom. Do not skip cells.

### Target Metrics
| Priority | Metric | Target |
| **PRIMARY** | Precision (Macro) | ≥ 0.80 |
| **PRIMARY** | Recall (Macro) | ≥ 0.80 |
| **PRIMARY** | F1 Macro | ≥ 0.80 |

### Architecture Summary
```
Input (224×224 RGB)
       ↓

### GPU Compute Update
This version automatically checks for GPU availability, enables TensorFlow GPU memory growth, enables mixed precision on compatible GPUs, and runs model creation/training inside a TensorFlow distribution strategy.



---
# PHASE 1: Environment Setup

**What we are doing:**
Installing the extra tools that Colab does not have by default,
importing everything we need, and verifying the GPU is working.
This is like checking your tools before starting a job.


In [6]:
# ── CELL 1: Install Required Libraries ──────────────────────────────────────
print('Installing required libraries...')

!pip install scipy --quiet

print('✅ All libraries installed.')
print()
print('⚠️  RESTART CHECK:')
print('   If this is a FRESH Colab session (just opened), restart the runtime:')
print('   Runtime → Restart session → Then re-run from Cell 0.')
print('   If you are CONTINUING a previous session, skip the restart.')

Installing required libraries...
✅ All libraries installed.

⚠️  RESTART CHECK:
   If this is a FRESH Colab session (just opened), restart the runtime:
   Runtime → Restart session → Then re-run from Cell 0.
   If you are CONTINUING a previous session, skip the restart.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB4
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from PIL import Image, UnidentifiedImageError
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── GPU / Accelerator Configuration ────────────────────────────────────────
# In Colab: Runtime → Change runtime type → Hardware accelerator → GPU.
# In Kaggle: Notebook settings → Accelerator → GPU.
# TensorFlow will use the GPU automatically once a compatible GPU runtime is active.
SEED = 42
tf.keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices('GPU')
if GPUS:
    print(f'✅ GPU detected: {len(GPUS)} device(s)')
    for gpu in GPUS:
        print('  ', gpu)
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print(f'  Could not set memory growth for {gpu}: {exc}')
else:
    print('⚠️ No GPU detected. Training will run on CPU unless you enable a GPU runtime.')

# Mixed precision speeds up EfficientNet training on NVIDIA T4 / P100 / V100 / A100 GPUs.
# Keep the final softmax layer in float32 later for stable losses/metrics.
USE_MIXED_PRECISION = bool(GPUS)
if USE_MIXED_PRECISION:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print('✅ Mixed precision enabled:', tf.keras.mixed_precision.global_policy())
    except Exception as exc:
        USE_MIXED_PRECISION = False
        print('⚠️ Mixed precision could not be enabled:', exc)
else:
    tf.keras.mixed_precision.set_global_policy('float32')

# MirroredStrategy supports single-GPU and multi-GPU training.
STRATEGY = tf.distribute.MirroredStrategy() if GPUS else tf.distribute.get_strategy()
print('✅ TensorFlow strategy:', type(STRATEGY).__name__)
print('✅ Replicas in sync:', STRATEGY.num_replicas_in_sync)

# Optional XLA JIT. This can improve speed, but disable it if your runtime throws XLA errors.
USE_XLA = False
if USE_XLA:
    tf.config.optimizer.set_jit(True)
    print('✅ XLA JIT enabled')

IMG_SIZE    = 224
BATCH_SIZE  = 32 * STRATEGY.num_replicas_in_sync
NUM_CLASSES = 5
THRESHOLD   = 0.5
SMOOTHING   = 0.1
AUTOTUNE    = tf.data.AUTOTUNE

CLASS_NAMES = [
    'Healthy', 'CMD', 'CBB', 'CGM', 'CBSD'
]

PROJECT_DIR = os.path.join(os.getcwd(), 'cassava_fyp')
os.makedirs(PROJECT_DIR, exist_ok=True)

print('✅ Imports successful.')
print(f'Project directory: {PROJECT_DIR}')
print(f'Batch size: {BATCH_SIZE}')


: 

In [ ]:
# ── GPU Runtime Verification ───────────────────────────────────────────────
print('TensorFlow version:', tf.__version__)
print('Physical GPUs:', tf.config.list_physical_devices('GPU'))
print('Logical GPUs:', tf.config.list_logical_devices('GPU'))
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())

# Small matrix multiplication test. If a GPU is active, TensorFlow places this op on GPU automatically.
with tf.device('/GPU:0' if tf.config.list_logical_devices('GPU') else '/CPU:0'):
    a = tf.random.normal((1024, 1024))
    b = tf.random.normal((1024, 1024))
    c = tf.matmul(a, b)
print('Compute test complete. Output shape:', c.shape)


: 

---
# PHASE 2: Data Collection

**What we are doing:**
Downloading the Kaggle Cassava Leaf Disease dataset.
This is your primary data source as stated in Chapter 3 Section 3.2A.

**Before running the next cell:**
1. Go to [kaggle.com](https://www.kaggle.com) → Account → Settings → API → Create New Token
2. This downloads a file called `kaggle.json` to your computer
3. Run the cell — it will prompt you to upload that file


In [ ]:
# ── CELL 3: Configure Kaggle API (Colab or Local Jupyter)
%pip install -q kaggle
import os, sys, shutil, stat
colab = 'google.colab' in sys.modules
paths = [
    './kaggle.json',
    os.path.expanduser('~/kaggle.json'),
    '/content/kaggle.json',
]
found = None
for p in paths:
    if p and os.path.exists(p):
        found = p
        break
if found:
    dst_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(found, os.path.join(dst_dir, 'kaggle.json'))
    os.chmod(os.path.join(dst_dir, 'kaggle.json'), 0o600)
    print(f'Copied {found} to ~/.kaggle/kaggle.json')
else:
    if colab:
        from google.colab import files
        print('Please upload your kaggle.json file when prompted.')
        uploaded = files.upload()
        for name in uploaded:
            open('/root/.kaggle/kaggle.json','wb').write(uploaded[name])
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print('Kaggle API key configured via upload.')
    else:
        print('kaggle.json not found. Place kaggle.json in the notebook folder or home directory and re-run this cell.')

print('✅ Kaggle API key configured (if kaggle.json was found).')


---
# PHASE 2: Data Collection, EDA, and Cleaning

This section downloads the Kaggle dataset using your `kaggle.json`,
inspects the image folders, removes unreadable files, and prepares a
clean metadata table for preprocessing and training.


In [ ]:
# Use existing local data folder or the cleaned metadata file (no download).
import os, glob, zipfile
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import pandas as pd

metadata_path = os.path.join(PROJECT_DIR, 'data', 'processed', 'clean_metadata.csv')
csv_root = None
if os.path.exists(metadata_path):
    preview_df = pd.read_csv(metadata_path)
    if 'image_path' in preview_df.columns and not preview_df.empty:
        existing_paths = [p for p in preview_df['image_path'].astype(str) if os.path.exists(p)]
        if existing_paths:
            csv_root = os.path.commonpath(existing_paths)
            if os.path.isfile(csv_root):
                csv_root = os.path.dirname(csv_root)

# Candidate locations to look for the dataset 'data' folder
candidates = [
    csv_root,
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), '..', 'data'),
    os.path.join(PROJECT_DIR, '..', 'data'),
    os.path.join(PROJECT_DIR, 'data', 'raw', 'data'),
    os.path.join(PROJECT_DIR, 'data'),
    os.path.join(os.getcwd(), 'cassava_fyp', 'data'),
]
DATA_DIR = None
for c in candidates:
    if not c:
        continue
    if os.path.exists(c) and any(fn.lower().endswith(('.jpg','.jpeg','.png')) for fn in glob.glob(os.path.join(c, '**', '*'), recursive=True)):
        DATA_DIR = c
        break

if DATA_DIR is None:
    if os.path.exists(metadata_path):
        raise FileNotFoundError('Found clean_metadata.csv, but none of its image_path entries exist on disk. Restore the original image folders or update the paths in the CSV, then re-run.')
    raise FileNotFoundError('Could not find a local `data` folder with images. Place the dataset under ./data, ../data, or cassava_fyp/data and re-run.')

print('Using DATA_DIR =', DATA_DIR)

# The Kaggle dataset uses folder names like 'Cassava___healthy' etc.
# We search for those folders under DATA_DIR; if not found, search for any image subfolders
expected_folders = {
    'Cassava___healthy': 'Healthy',
    'Cassava___mosaic_disease': 'CMD',
    'Cassava___bacterial_blight': 'CBB',
    'Cassava___green_mottle': 'CGM',
    'Cassava___brown_streak_disease': 'CBSD',
}
records = []
# If expected folders exist under DATA_DIR/data, use them; otherwise find subfolders with images
for folder_name, label in expected_folders.items():
    folder_path = os.path.join(DATA_DIR, folder_name)
    if os.path.exists(folder_path):
        image_paths = sorted(glob.glob(os.path.join(folder_path, '*')))
        for p in image_paths:
            if p.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({'image_path': p, 'label': label})

# Fallback: if no records yet, find any subfolder containing images and try to infer labels from folder names
if not records:
    for sub in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
        if os.path.isdir(sub):
            imgs = [p for p in glob.glob(os.path.join(sub, '*')) if p.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                label = os.path.basename(sub)
                # map common folder names to canonical labels when possible
                mapped = None
                for k, v in expected_folders.items():
                    if k.lower() in label.lower() or v.lower() == label.lower():
                        mapped = v; break
                if mapped is None:
                    mapped = label
                for p in imgs:
                    records.append({'image_path': p, 'label': mapped})

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError(f'No images found under {DATA_DIR} — check your data layout.')

print(f'Found {len(df):,} images before cleaning.')
print(df['label'].value_counts().sort_index())

# Basic EDA plot
plt.figure(figsize=(8, 4))
counts = df['label'].value_counts().reindex(sorted(df['label'].unique()))
plt.bar(counts.index, counts.values, color='#2E86C1')
plt.title('Cassava class distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Cleaning: remove unreadable images and duplicates using PIL
clean_rows = []
bad_files = []
seen_paths = set()
for row in df.itertuples(index=False):
    p = row.image_path
    if p in seen_paths:
        continue
    seen_paths.add(p)
    try:
        with Image.open(p) as img:
            img.verify()
        clean_rows.append({'image_path': p, 'label': row.label})
    except Exception:
        bad_files.append(p)

df_clean = pd.DataFrame(clean_rows)
print(f'Clean images: {len(df_clean):,}')
print(f'Removed unreadable images: {len(bad_files):,}')

os.makedirs(os.path.join(PROJECT_DIR, 'data', 'processed'), exist_ok=True)
out_path = os.path.join(PROJECT_DIR, 'data', 'processed', 'clean_metadata.csv')
df_clean.to_csv(out_path, index=False)
print('Saved clean metadata to', out_path)

# Show a few sample images
sample_df = df_clean.groupby('label').head(2).reset_index(drop=True)
ncols = min(5, len(sample_df))
fig, axes = plt.subplots(2, ncols, figsize=(4*ncols, 6))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for ax, row in zip(axes, sample_df.itertuples(index=False)):
    with Image.open(row.image_path) as image:
        ax.imshow(image)
    ax.set_title(row.label)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
df_clean = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'processed', 'clean_metadata.csv'))
label_to_idx = {'Healthy': 0, 'CMD': 1, 'CBB': 2, 'CGM': 3, 'CBSD': 4}

df_clean['label_idx'] = df_clean['label'].map(label_to_idx)
if df_clean['label_idx'].isna().any():
    unknown_labels = sorted(df_clean.loc[df_clean['label_idx'].isna(), 'label'].unique().tolist())
    raise ValueError(f'Found labels not covered by label_to_idx: {unknown_labels}')
df_clean['label_idx'] = df_clean['label_idx'].astype(int)

train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=SEED,
    stratify=df_clean['label_idx'],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['label_idx'],
)

print('Split sizes:')
print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")
print()
print('Train class counts:')
print(train_df['label'].value_counts().sort_index())
print()
print('Val class counts:')
print(val_df['label'].value_counts().sort_index())
print()
print('Test class counts:')
print(test_df['label'].value_counts().sort_index())


def preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bicubic')
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img


def augment_image(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.12)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.image.random_saturation(image, 0.85, 1.15)
    return image, label


def _dataset_options():
    options = tf.data.Options()
    options.experimental_deterministic = False
    return options


def make_image_dataset(frame, training=False, batch_size=BATCH_SIZE):
    paths = frame['image_path'].values
    labels = frame['label_idx'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.with_options(_dataset_options())
    ds = ds.map(lambda p, y: (preprocess_image(p), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
    return ds


def make_balanced_train_dataset(frame, batch_size=BATCH_SIZE):
    class_frames = [frame[frame['label_idx'] == class_index] for class_index in range(NUM_CLASSES)]
    max_class_size = max(len(class_frame) for class_frame in class_frames)
    balanced_parts = []
    for class_frame in class_frames:
        if class_frame.empty:
            continue
        repeats = int(np.ceil(max_class_size / len(class_frame)))
        repeated_frame = pd.concat([class_frame] * repeats, ignore_index=True)
        repeated_frame = repeated_frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        balanced_parts.append(repeated_frame.iloc[:max_class_size].copy())
    balanced_frame = pd.concat(balanced_parts, ignore_index=True)
    balanced_frame = balanced_frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return make_image_dataset(balanced_frame, training=True, batch_size=batch_size)


In [ ]:
import hashlib
from collections import Counter


def file_sha1(path, chunk_size=1024 * 1024):
    hasher = hashlib.sha1()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            hasher.update(chunk)
    return hasher.hexdigest()


def split_leak_report(train_frame, val_frame, test_frame):
    splits = {
        'train': train_frame,
        'val': val_frame,
        'test': test_frame,
    }

    print('Leak check: exact path overlap')
    path_sets = {name: set(frame['image_path'].astype(str)) for name, frame in splits.items()}
    for left_name in splits:
        for right_name in splits:
            if left_name >= right_name:
                continue
            overlap = path_sets[left_name] & path_sets[right_name]
            print(f'  {left_name} vs {right_name}: {len(overlap)} shared paths')

    print()
    print('Leak check: duplicate filenames across splits')
    stem_sets = {
        name: set(frame['image_path'].astype(str).map(lambda p: os.path.basename(p)))
        for name, frame in splits.items()
    }
    for left_name in splits:
        for right_name in splits:
            if left_name >= right_name:
                continue
            overlap = stem_sets[left_name] & stem_sets[right_name]
            print(f'  {left_name} vs {right_name}: {len(overlap)} shared filenames')

    print()
    print('Leak check: exact file-content duplicates via SHA1')
    hash_sets = {}
    for name, frame in splits.items():
        hashes = []
        missing = 0
        for path in frame['image_path'].astype(str):
            if os.path.exists(path):
                hashes.append(file_sha1(path))
            else:
                missing += 1
        hash_sets[name] = set(hashes)
        print(f'  {name}: {len(hashes):,} hashed, {missing:,} missing paths')

    for left_name in splits:
        for right_name in splits:
            if left_name >= right_name:
                continue
            overlap = hash_sets[left_name] & hash_sets[right_name]
            print(f'  {left_name} vs {right_name}: {len(overlap)} shared image hashes')

    print()
    print('Label distribution by split')
    for name, frame in splits.items():
        print(f'  {name}:')
        print(frame['label'].value_counts().sort_index())
        print()


split_leak_report(train_df, val_df, test_df)


In [ ]:
def msa_block(feature_map):
    channels = int(feature_map.shape[-1])
    squeeze = layers.GlobalAveragePooling2D()(feature_map)
    squeeze = layers.Dense(max(channels // 8, 1), activation='relu')(squeeze)
    squeeze = layers.Dense(channels, activation='sigmoid')(squeeze)
    squeeze = layers.Reshape((1, 1, channels))(squeeze)
    channel_refined = layers.Multiply()([feature_map, squeeze])

    spatial = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(channel_refined)
    return layers.Multiply()([channel_refined, spatial])


def build_model():
    backbone = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    x = msa_block(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    # Keep final probabilities in float32 for numerical stability under mixed precision.
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    model = Model(inputs, outputs, name='cassava_efficientnetb4_msa')
    return model, backbone


with STRATEGY.scope():
    model, backbone = build_model()

print('Model summary:')
model.summary()
print('✅ Model created inside strategy scope. Device placement will use GPU when available.')


In [ ]:
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, 'model', 'checkpoints', 'best_model.keras')
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

train_ds = make_balanced_train_dataset(train_df)
val_ds = make_image_dataset(val_df, training=False)
test_ds = make_image_dataset(test_df, training=False)

# Ensure SMOOTHING is defined in case the kernel lost earlier state
if 'SMOOTHING' not in globals():
    SMOOTHING = 0.1


def smooth_sparse_categorical_crossentropy(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_true = tf.one_hot(y_true, NUM_CLASSES)
    y_true = y_true * (1.0 - SMOOTHING) + (SMOOTHING / NUM_CLASSES)
    return tf.keras.losses.categorical_crossentropy(y_true, y_pred)


with STRATEGY.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=smooth_sparse_categorical_crossentropy,
        metrics=['accuracy'],
        jit_compile=USE_XLA,
    )

print('Training device check:', tf.config.list_logical_devices('GPU'))

training_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
]

print('Stage 1: frozen backbone')
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=training_callbacks,
    verbose=1,
)

backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False

with STRATEGY.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),
        loss=smooth_sparse_categorical_crossentropy,
        metrics=['accuracy'],
        jit_compile=USE_XLA,
    )

print('Stage 2: fine-tune top backbone layers')
finetune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=training_callbacks,
    verbose=1,
)

os.makedirs(os.path.join(PROJECT_DIR, 'model'), exist_ok=True)
model.save(os.path.join(PROJECT_DIR, 'model', 'final_model.keras'))

import joblib
joblib.dump(
    {'class_names': CLASS_NAMES, 'sampling': 'balanced_oversampling', 'label_smoothing': SMOOTHING},
    os.path.join(PROJECT_DIR, 'model', 'training_metadata.joblib'),
)


In [ ]:
# Evaluation on validation and held-out test sets
val_ds = make_image_dataset(val_df, training=False)
test_ds = make_image_dataset(test_df, training=False)

val_prob = model.predict(val_ds, verbose=0)
val_pred = np.argmax(val_prob, axis=1)
val_true = val_df['label_idx'].values.astype(int)

val_acc = accuracy_score(val_true, val_pred)
val_f1 = f1_score(val_true, val_pred, average='macro', zero_division=0)
val_prec = precision_score(val_true, val_pred, average='macro', zero_division=0)
val_rec = recall_score(val_true, val_pred, average='macro', zero_division=0)

print('Validation metrics:')
print(f'Accuracy:        {val_acc:.4f}')
print(f'F1 Macro:        {val_f1:.4f}')
print(f'Precision Macro: {val_prec:.4f}')
print(f'Recall Macro:    {val_rec:.4f}')
print()

y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_df['label_idx'].values.astype(int)

test_acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)

print('Final metrics:')
print(f'Accuracy:        {test_acc:.4f}')
print(f'F1 Macro:        {f1_macro:.4f}')
print(f'Precision Macro: {prec_macro:.4f}')
print(f'Recall Macro:    {rec_macro:.4f}')

results = pd.DataFrame([
    {'metric': 'accuracy', 'value': test_acc},
    {'metric': 'f1_macro', 'value': f1_macro},
    {'metric': 'precision_macro', 'value': prec_macro},
    {'metric': 'recall_macro', 'value': rec_macro},
])
results.to_csv(os.path.join(PROJECT_DIR, 'results', 'metrics', 'evaluation_metrics.csv'), index=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=results, x='metric', y='value')
plt.ylim(0, 1)
plt.title('Evaluation Metrics after EfficientNetB4 + MSA Training')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
val_ds = make_image_dataset(val_df, training=False)
val_prob = model.predict(val_ds, verbose=0)
val_pred = np.argmax(val_prob, axis=1)
val_true = val_df['label_idx'].values.astype(int)

print('Validation classification report:')
print(classification_report(val_true, val_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(val_true, val_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Validation Confusion Matrix - EfficientNetB4 + MSA')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

per_class_recall = recall_score(val_true, val_pred, average=None, zero_division=0)
per_class_precision = precision_score(val_true, val_pred, average=None, zero_division=0)
print('Per-class validation precision/recall:')
for class_name, p, r in zip(CLASS_NAMES, per_class_precision, per_class_recall):
    print(f'{class_name:10s}  precision={p:.4f}  recall={r:.4f}')